# 🛍️ Stabraq RAG Chatbot — Colab T4 Optimised
**Stack:** LangChain · Groq · Pinecone · HuggingFace Embeddings · Gradio

### What changed from v1:
| Problem | Fix |
|---|---|
| 1112 pages scraped serially (~3 hrs) | Async scraping with `asyncio` + `aiohttp` (10× faster) + skip `/ar/` duplicates |
| 418k chunks → OOM | Larger chunks (1200), no overlap, strict dedup, embed+upsert in streaming batches (never hold all chunks in RAM) |
| T4 GPU unused | GPU used for embedding with batch_size=128 |

## 1. Install Dependencies

In [10]:
# Step 1: Uninstall the broken versions and install compatible ones
# After this cell finishes → Restart Runtime (Runtime menu → Restart session)
!pip install -q --upgrade pip
!pip uninstall -y numpy scipy scikit-learn sentence-transformers
!pip install -q \
    "numpy==1.26.4" \
    "scipy==1.13.1" \
    "scikit-learn==1.4.2" \
    "sentence-transformers==3.0.1" \
    langchain langchain-text-splitters \
    langchain-groq langchain-pinecone langchain-huggingface \
    pinecone-client \
    aiohttp requests beautifulsoup4 lxml gradio

print("\n✅ Done — NOW go to Runtime → Restart session, then run from Cell 2 onwards")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.8 MB/s eta 0:00:00
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: scipy 1.16.3
Uninstalling scipy-1.16.3:
  Successfully uninstalled scipy-1.16.3
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: sentence-transformers 2.7.0
Uninstalling sentence-transformers-2.7.0:
  Successfully uninstalled sentence-transformers-2.7.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9

In [2]:
import numpy as np
import sentence_transformers
from langchain_huggingface import HuggingFaceEmbeddings
print(f"numpy:               {np.__version__}")
print(f"sentence_transformers: {sentence_transformers.__version__}")
print("✅ All imports clean — proceed with the rest of the notebook")

numpy:               1.26.4
sentence_transformers: 3.0.1
✅ All imports clean — proceed with the rest of the notebook


## 2. Configuration

In [3]:
import os

GROQ_API_KEY     = "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"
PINECONE_API_KEY = "pcsk_7WTnyf_2aJZg2NSqkX4Y5Qj5HLPzwucEyvvwbJMWANv2XQXM9kqczXf6Q7hQEoDGoghxFW"

PINECONE_INDEX  = "stabraq-rag"
PINECONE_REGION = "us-east-1"
EMBEDDING_DIM   = 768

EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
GROQ_MODEL  = "openai/gpt-oss-120b"

# ── Larger chunks = far fewer of them = less RAM ──────────────
CHUNK_SIZE    = 1200   # was 800
CHUNK_OVERLAP = 0      # was 100 — overlap multiplies chunk count, skip it

# ── Scraping ──────────────────────────────────────────────────
CONCURRENT_REQUESTS = 15   # async concurrency limit — polite but fast
SCRAPE_TIMEOUT      = 15   # seconds per request

# ── Embedding ─────────────────────────────────────────────────
EMBED_BATCH   = 128    # T4 can handle 128 at once comfortably
UPSERT_BATCH  = 100    # Pinecone upsert batch

os.environ["GROQ_API_KEY"]     = GROQ_API_KEY
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

from pinecone import Pinecone
try:
    pc = Pinecone(api_key=PINECONE_API_KEY)
    pc.list_indexes()
    print("✅ Pinecone key valid")
except Exception as e:
    print(f"❌ Pinecone auth failed: {e}")

✅ Pinecone key valid


## 3. Parse Sitemaps → Collect URLs

**Key change:** We only scrape **English URLs** and skip the `/ar/` mirror sitemaps entirely.  
The Arabic content is nearly identical — scraping both was doubling page count and chunk count with minimal retrieval benefit.  
The multilingual embedding model still handles Arabic *queries* perfectly; we just don't duplicate the source data.

In [4]:
import requests
import time
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from collections import Counter

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; StabraqBot/1.0)"}

# English-only sitemaps (cuts URL count in half vs v1)
CHILD_SITEMAPS = [
    "https://stabraq.com/sitemap_products_1.xml?from=6993635573955&to=9192011464954",
    "https://stabraq.com/sitemap_pages_1.xml?from=21906001&to=127013683450",
    "https://stabraq.com/sitemap_collections_1.xml?from=276434419907&to=468571455738",
    "https://stabraq.com/sitemap_blogs_1.xml",
    "https://stabraq.com/sitemap_metaobject_pages_1.xml",
]

def parse_sitemap(url: str) -> list[dict]:
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
    except Exception as e:
        print(f"  ⚠️  {url}: {e}")
        return []
    soup = BeautifulSoup(resp.text, "xml")
    entries = []
    for loc_tag in soup.find_all("loc"):
        loc  = loc_tag.get_text(strip=True)
        path = urlparse(loc).path
        if   "/products/"   in path: ptype = "product"
        elif "/collections/" in path: ptype = "collection"
        elif "/blogs/"       in path or "/articles/" in path: ptype = "blog"
        else: ptype = "page"
        entries.append({"url": loc, "type": ptype})
    return entries

all_url_entries = []
for sm in CHILD_SITEMAPS:
    entries = parse_sitemap(sm)
    print(f"  {sm.split('stabraq.com')[1][:55]:<55} → {len(entries)} URLs")
    all_url_entries.extend(entries)
    time.sleep(0.3)

# Deduplicate
seen, unique_entries = set(), []
for e in all_url_entries:
    if e["url"] not in seen:
        seen.add(e["url"])
        unique_entries.append(e)

print(f"\n✅ {len(unique_entries)} unique URLs | {dict(Counter(e['type'] for e in unique_entries))}")

  /sitemap_products_1.xml?from=6993635573955&to=919201146 → 497 URLs
  /sitemap_pages_1.xml?from=21906001&to=127013683450      → 21 URLs
  /sitemap_collections_1.xml?from=276434419907&to=4685714 → 167 URLs
  /sitemap_blogs_1.xml                                    → 4 URLs
  /sitemap_metaobject_pages_1.xml                         → 1 URLs

✅ 689 unique URLs | {'page': 257, 'product': 261, 'collection': 167, 'blog': 4}


## 4. Async Scraping — 10× Faster Than Serial

**Root cause of 3-hour runtime:** The original loop did `requests.get()` one page at a time with a 0.5s sleep → ~10 min/100 pages.  

**Fix:** `aiohttp` with a semaphore-bounded concurrency of 15 parallel requests.  
All 500–600 pages scrape in **3–5 minutes** instead of hours.  
We also extract only the **JSON-LD block** for products (the cleanest, most compact signal) and skip the full HTML parse for those — dramatically reducing per-document text size.

In [5]:
import asyncio
import aiohttp
import json
import re
from bs4 import BeautifulSoup
from langchain_core.documents import Document

NOISE_TAGS = ["script", "style", "noscript", "svg", "nav",
              "footer", "header", "aside", "form", "button", "iframe"]

CONTENT_SELECTORS = {
    "product":    [".product__description", ".product-description",
                   "#description", ".description", ".rte"],
    "collection": [".collection-description", ".collection__description", ".rte"],
    "blog":       ["article", ".article__body", ".article-body", ".rte"],
    "page":       [".page-content", ".page__content", ".rte", "main"],
}


def extract_json_ld(soup: BeautifulSoup) -> str:
    """Pull clean structured data from Shopify's JSON-LD injection."""
    parts = []
    for tag in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(tag.string or "{}")
            for key in ["name", "description", "price", "brand", "sku"]:
                val = data.get(key)
                if isinstance(val, str) and val.strip():
                    parts.append(f"{key}: {val.strip()}")
            offers = data.get("offers", {})
            if isinstance(offers, dict):
                for k in ["price", "priceCurrency", "availability"]:
                    if v := offers.get(k):
                        parts.append(f"{k}: {v}")
        except Exception:
            pass
    return "\n".join(parts)


def parse_html(html: str, page_type: str) -> str:
    soup = BeautifulSoup(html, "lxml")
    h1   = soup.find("h1")
    title = h1.get_text(strip=True) if h1 else ""

    # For products: JSON-LD is cleaner and more compact than full HTML
    if page_type == "product":
        jld = extract_json_ld(soup)
        # Also grab the description block if present
        for tag in soup(NOISE_TAGS): tag.decompose()
        desc = ""
        for sel in CONTENT_SELECTORS["product"]:
            el = soup.select_one(sel)
            if el:
                desc = el.get_text(separator=" ", strip=True)
                break
        return "\n".join(filter(None, [title, jld, desc]))

    # For other types: strip noise then pull targeted selectors
    for tag in soup(NOISE_TAGS): tag.decompose()
    selectors = CONTENT_SELECTORS.get(page_type, CONTENT_SELECTORS["page"])
    parts, seen = [], set()
    for sel in selectors:
        for el in soup.select(sel):
            t = el.get_text(separator=" ", strip=True)
            if t and t not in seen:
                parts.append(t)
                seen.add(t)
    if not parts:
        fb = soup.find("main") or soup.find("body")
        if fb: parts.append(fb.get_text(separator=" ", strip=True))
    return "\n".join(filter(None, [title] + parts))


async def fetch_one(session, entry, semaphore):
    async with semaphore:
        try:
            async with session.get(entry["url"], timeout=aiohttp.ClientTimeout(total=SCRAPE_TIMEOUT)) as resp:
                html = await resp.text(errors="replace")
            text = parse_html(html, entry["type"])
            if len(text) >= 80:
                return Document(
                    page_content=text,
                    metadata={"source": entry["url"], "type": entry["type"]},
                )
        except Exception:
            pass
    return None


async def scrape_all(entries):
    semaphore = asyncio.Semaphore(CONCURRENT_REQUESTS)
    connector = aiohttp.TCPConnector(limit=CONCURRENT_REQUESTS)
    async with aiohttp.ClientSession(
        headers=HEADERS,
        connector=connector,
        connector_owner=True,
    ) as session:
        tasks = [fetch_one(session, e, semaphore) for e in entries]
        results = []
        for i, coro in enumerate(asyncio.as_completed(tasks)):
            doc = await coro
            if doc: results.append(doc)
            if (i + 1) % 50 == 0:
                print(f"  [{i+1}/{len(entries)}] {len(results)} docs scraped")
        return results


print(f"🚀 Async scraping {len(unique_entries)} pages (concurrency={CONCURRENT_REQUESTS})…")
t0 = time.time()
all_documents = await scrape_all(unique_entries)   # Colab supports top-level await
print(f"\n✅ {len(all_documents)} documents in {time.time()-t0:.0f}s")

🚀 Async scraping 689 pages (concurrency=15)…
  [50/689] 25 docs scraped
  [100/689] 55 docs scraped
  [150/689] 86 docs scraped
  [200/689] 121 docs scraped
  [250/689] 154 docs scraped
  [300/689] 193 docs scraped
  [350/689] 228 docs scraped
  [400/689] 259 docs scraped
  [450/689] 290 docs scraped
  [500/689] 323 docs scraped
  [550/689] 355 docs scraped
  [600/689] 382 docs scraped
  [650/689] 411 docs scraped

✅ 435 documents in 61s


## 5. Preprocess & Clean

Same cleaning logic as before, but we also **deduplicate near-identical documents** by hashing the first 300 chars.  
Shopify collection pages repeat the same product titles across multiple collection URLs — this was a major source of chunk inflation.

In [6]:
import hashlib

BOILERPLATE_RE = re.compile(
    r"Powered by Shopify"
    r"|Add to [Cc]art|اضف الى العربة|أضف إلى السلة"
    r"|Buy now|اشتري الآن"
    r"|Free shipping on orders over"
    r"|Subscribe to our newsletter|اشترك في نشرتنا البريدية"
    r"|We use cookies|نستخدم ملفات تعريف الارتباط|Accept [Cc]ookies"
    r"|All rights reserved|جميع الحقوق محفوظة"
    r"|\[.*?\]",
    re.IGNORECASE,
)

def clean_text(text: str) -> str:
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = BOILERPLATE_RE.sub("", text)
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"[\u0622\u0623\u0625]", "\u0627", text)  # Alef normalisation
    text = re.sub(r"\u0649", "\u064a", text)                 # Alef Maqsoura
    return text.strip()


cleaned_documents = []
seen_hashes = set()

for doc in all_documents:
    clean = clean_text(doc.page_content)
    if len(clean) < 80:
        continue
    # Near-duplicate detection: hash first 300 chars
    fingerprint = hashlib.md5(clean[:300].encode()).hexdigest()
    if fingerprint in seen_hashes:
        continue
    seen_hashes.add(fingerprint)
    cleaned_documents.append(Document(page_content=clean, metadata=doc.metadata))

print(f"Raw: {len(all_documents):,} → Cleaned & deduped: {len(cleaned_documents):,} documents")

Raw: 435 → Cleaned & deduped: 63 documents


## 6. Chunk Documents

**Why 418k chunks?** Chunk size 800 + overlap 100 on thousands of pages = massive count.  
**Fix:** Chunk size 1200, zero overlap. Fewer, denser chunks → fewer embeddings → less RAM and faster indexing.

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", "،", " ", ""],
)

chunks = splitter.split_documents(cleaned_documents)
print(f"✅ {len(chunks):,} chunks from {len(cleaned_documents):,} documents")
print(f"   Avg chunk length: {sum(len(c.page_content) for c in chunks)//len(chunks)} chars")

✅ 58,989 chunks from 63 documents
   Avg chunk length: 1063 chars


## 7. Embed & Index — Streaming to Pinecone (No OOM)

**Root cause of OOM:** The old loop called `PineconeVectorStore.from_documents(batch)` which internally embeds the entire batch *and* keeps the vectors in memory before upserting.  
When done across 418k chunks this fills the T4's 15GB VRAM and Colab's 12GB RAM.

**Fix:** Stream in pages of 500 chunks:
1. Embed page with GPU (batch_size=128 for T4)
2. Upsert directly to Pinecone
3. **Delete** the page from RAM before loading the next one

Peak RAM usage stays at ~1 page × chunk size instead of all chunks at once.

In [8]:
import gc
import time
from pinecone import Pinecone, ServerlessSpec
from langchain_huggingface import HuggingFaceEmbeddings

# ── Embedding model on GPU ────────────────────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cuda"},      # T4 GPU
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": EMBED_BATCH,        # 128 fits T4 comfortably
    },
)

# ── Pinecone index ────────────────────────────────────────────
pc = Pinecone(api_key=PINECONE_API_KEY)
if PINECONE_INDEX not in [idx.name for idx in pc.list_indexes()]:
    pc.create_index(
        name=PINECONE_INDEX,
        dimension=EMBEDDING_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region=PINECONE_REGION),
    )
    print(f"✅ Created index '{PINECONE_INDEX}'")
else:
    print(f"ℹ️  Index '{PINECONE_INDEX}' exists — reusing")

index = pc.Index(PINECONE_INDEX)

# ── Stream embed → upsert, one page at a time ─────────────────
PAGE_SIZE = 500   # chunks processed per iteration — keeps RAM flat
total_upserted = 0
t0 = time.time()

for page_start in range(0, len(chunks), PAGE_SIZE):
    page = chunks[page_start : page_start + PAGE_SIZE]

    # 1. Embed this page on GPU
    texts = [c.page_content for c in page]
    vectors = embeddings.embed_documents(texts)   # returns List[List[float]]

    # 2. Build Pinecone upsert payload and push in sub-batches
    for i in range(0, len(page), UPSERT_BATCH):
        sub_chunks  = page[i : i + UPSERT_BATCH]
        sub_vectors = vectors[i : i + UPSERT_BATCH]
        records = [
            {
                "id":     f"doc-{page_start + i + j}",
                "values": sub_vectors[j],
                "metadata": {
                    "text":   sub_chunks[j].page_content[:1000],  # Pinecone metadata limit
                    "source": sub_chunks[j].metadata.get("source", ""),
                    "type":   sub_chunks[j].metadata.get("type", ""),
                },
            }
            for j in range(len(sub_chunks))
        ]
        index.upsert(vectors=records)
        total_upserted += len(records)

    # 3. Free this page from RAM before loading the next one
    del page, texts, vectors
    gc.collect()

    elapsed = time.time() - t0
    print(f"  {total_upserted}/{len(chunks)} upserted | {elapsed:.0f}s elapsed")

print(f"\n✅ Done — {total_upserted:,} vectors in Pinecone")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ℹ️  Index 'stabraq-rag' exists — reusing
  500/58989 upserted | 12s elapsed
  1000/58989 upserted | 22s elapsed
  1500/58989 upserted | 32s elapsed
  2000/58989 upserted | 42s elapsed
  2500/58989 upserted | 52s elapsed
  3000/58989 upserted | 63s elapsed
  3500/58989 upserted | 73s elapsed
  4000/58989 upserted | 83s elapsed
  4500/58989 upserted | 93s elapsed
  5000/58989 upserted | 104s elapsed
  5500/58989 upserted | 114s elapsed
  6000/58989 upserted | 124s elapsed
  6500/58989 upserted | 134s elapsed
  7000/58989 upserted | 145s elapsed
  7500/58989 upserted | 155s elapsed
  8000/58989 upserted | 165s elapsed
  8500/58989 upserted | 175s elapsed
  9000/58989 upserted | 186s elapsed
  9500/58989 upserted | 196s elapsed
  10000/58989 upserted | 207s elapsed
  10500/58989 upserted | 217s elapsed
  11000/58989 upserted | 227s elapsed
  11500/58989 upserted | 238s elapsed
  12000/58989 upserted | 248s elapsed
  12500/58989 upserted | 258s elapsed
  13000/58989 upserted | 269s elapsed


## 8. Build the RAG Chain

In [9]:
from langchain_groq import ChatGroq
from langchain_pinecone import PineconeVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

# Reload embeddings (safe to re-run from here without re-indexing)
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": EMBED_BATCH},
)

vectorstore = PineconeVectorStore(
    index_name=PINECONE_INDEX,
    embedding=embeddings,
    text_key="text",       # must match metadata key used during upsert
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0.2,
    api_key=GROQ_API_KEY,
)

PROMPT_TEMPLATE = """\
You are a helpful shopping assistant for Stabraq (استبرق), a clothing brand.
Use ONLY the information in the context below to answer the question.
If the answer is not in the context, say "I don't have that information" \
(or in Arabic: "لا تتوفر لديّ هذه المعلومات").
Reply in the same language the customer used.

Context:
{context}

Customer question: {question}

Answer:"""

prompt = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=["context", "question"],
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt},
)

print("✅ RAG chain ready")

/tmp/ipykernel_3224/3212761874.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_3224/3212761874.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


✅ RAG chain ready


## 9. Gradio Chat Interface

In [10]:
import gradio as gr

def chat(message: str, history: list):
    if not message.strip():
        return "", history
    result = qa_chain.invoke({"query": message})
    answer = result["result"]
    sources = list({
        doc.metadata.get("source", "")
        for doc in result["source_documents"]
        if doc.metadata.get("source")
    })[:3]
    if sources:
        answer += "\n\n🔗 " + " | ".join(sources)
    history.append((message, answer))
    return "", history


with gr.Blocks(title="Stabraq Chatbot", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """# 🛍️ Stabraq Assistant / مساعد استبرق
Ask me anything about products, collections, sizes, and more.
اسألني عن المنتجات والمجموعات والمقاسات وغير ذلك."""
    )
    chatbot = gr.Chatbot(height=450, bubble_full_width=False)
    msg     = gr.Textbox(placeholder="Ask a question… / اكتب سؤالك…", label="", lines=1)
    with gr.Row():
        send_btn  = gr.Button("Send / إرسال", variant="primary")
        clear_btn = gr.ClearButton([msg, chatbot], value="Clear / مسح")
    gr.Examples(
        examples=[
            "What collections do you have?",
            "ما هي المقاسات المتاحة؟",
            "Do you offer free shipping?",
            "أخبرني عن منتجاتكم الجديدة",
            "What is your return policy?",
        ],
        inputs=msg,
    )
    send_btn.click(chat, [msg, chatbot], [msg, chatbot])
    msg.submit(chat, [msg, chatbot], [msg, chatbot])

demo.launch(share=True)

/tmp/ipykernel_3224/3595096406.py:19: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Stabraq Chatbot", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_3224/3595096406.py:25: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=450, bubble_full_width=False)
/tmp/ipykernel_3224/3595096406.py:25: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(height=450, bubble_full_width=False)
/tmp/ipykernel_3224/3595096406.py:25: DeprecationWarning: The default value of 'allow_tags' in gr.Chatb

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6ffa50e5c51f2cb55e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
